# Day 8: Understanding Embeddings

## What is an Embedding?
Text → list of numbers (vector) where **similar meanings = close vectors**

- "Python Developer"  → [0.23, -0.11, 0.87, ...]
- "Python Engineer"   → [0.24, -0.10, 0.85, ...]  ← very close!
- "Graphic Designer"  → [-0.52, 0.63, -0.21, ...] ← far away

Similarity measured with **cosine similarity**: ranges -1 to 1
- 1.0 = identical meaning
- 0.0 = unrelated
- -1.0 = opposite

In [ ]:
# Verify all installations
import importlib

libs = {
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "chromadb": "chromadb",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}

for module, package in libs.items():
    try:
        importlib.import_module(module)
        print(f"✅ {package}")
    except ImportError:
        print(f"❌ {package} — run: pip install {package}")

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading model (downloads ~90MB first time)...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Model loaded!")
print(f"   Embedding dimensions: {model.get_sentence_embedding_dimension()}")

In [ ]:
import numpy as np

skill_phrases = [
    "Python developer with machine learning experience",
    "Python engineer skilled in data science",
    "Java backend developer",
    "React frontend developer",
    "Machine learning specialist with deep learning",
    "Senior Android developer with Kotlin",
    "SQL database administrator",
    "DevOps engineer with AWS and Kubernetes",
    "Data engineer with Spark and Airflow",
    "NLP researcher with transformer models",
]

print("Encoding phrases...")
embeddings = model.encode(skill_phrases, show_progress_bar=True)

print(f"\n✅ Shape: {embeddings.shape}")
print(f"   {len(skill_phrases)} phrases × {embeddings.shape[1]} dimensions")
print(f"\nFirst vector (first 8 values): {embeddings[0][:8].round(4)}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sim_matrix = cosine_similarity(embeddings)

short_labels = [
    "Python+ML", "Python+DS", "Java Backend", "React Frontend",
    "ML Specialist", "Android Dev", "SQL DBA", "DevOps+AWS",
    "Data Engineer", "NLP Research"
]

sim_df = pd.DataFrame(sim_matrix.round(2),
                       index=short_labels,
                       columns=short_labels)
print("Cosine Similarity Matrix:")
print(sim_df.to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    sim_matrix, annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=0, vmax=1,
    xticklabels=short_labels,
    yticklabels=short_labels, ax=ax,
    linewidths=0.5
)
ax.set_title("Skill Embedding Similarity Heatmap", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../data/processed/day8_similarity_heatmap.png", dpi=150)
plt.show()
print("✅ Saved heatmap")

In [ ]:
# Simulate a real query: find best candidate for a project role
project_requirement = "We need a Python developer experienced in machine learning and data pipelines"

candidates = [
    "E001: Python, TensorFlow, scikit-learn, pandas, data pipelines, AWS",
    "E002: Java, Spring Boot, microservices, REST APIs, PostgreSQL",
    "E003: Python, pandas, SQL, ETL, Airflow, data warehousing",
    "E004: React, JavaScript, TypeScript, CSS, Node.js",
    "E005: Python, PyTorch, deep learning, NLP, transformers, FAISS",
    "E006: AWS, Docker, Kubernetes, CI/CD, Terraform, Linux",
]

req_vec  = model.encode([project_requirement])
cand_vec = model.encode(candidates)

scores = cosine_similarity(req_vec, cand_vec)[0]
ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

print(f"Requirement: '{project_requirement}'\n")
print("=" * 65)
for rank, (profile, score) in enumerate(ranked, 1):
    bar = "█" * int(score * 30)
    print(f"#{rank} [{score:.4f}] {bar}")
    print(f"    {profile}\n")

## ✅ Day 8 Takeaways
1. `all-MiniLM-L6-v2` → 384-dim vectors, fast, good quality
2. Cosine similarity catches semantic matches keyword search misses
3. "Python+ML" ↔ "Python+DS" score HIGH — correct
4. "Java Backend" ↔ "React Frontend" score LOW — correct
5. Tomorrow: embed all 80 employees + 30 projects from our real CSVs